In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from supabase import create_client, Client
supabase_url = os.environ.get("SUPABASE_URL")
supabase_api_key = os.environ.get("SUPBASE_KEY")

supabase: Client = create_client(supabase_url, supabase_api_key)

In [2]:
from openai import OpenAI
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [21]:
def report_research(date, next_date):
    result = supabase.table('curation_selected_items')\
        .select('event_id, output,sources, news_date','topic, created_at')\
    .gte('created_at', date)\
    .lt('created_at', next_date)\
    .eq('topic', 'Report')\
        .order('created_at', desc=False)\
        .execute()
    
    return result.data

In [30]:
report_deep_dive = report_research('2026-02-27', '2026-02-28')

In [31]:
report_deep_dive

[{'event_id': '1633_2026-02-25',
  'output': 'Multiple reputable outlets (NBC News, Al Jazeera, Deutsche Welle — all reporting Feb 25, 2026) reported that US Defense Secretary Pete Hegseth gave Anthropic CEO Dario Amodei an ultimatum to allow the Pentagon unrestricted use of Anthropic models for all lawful military purposes by Friday (reported deadline), or face actions including being labeled a supply-chain risk or compelled under the Defense Production Act. Anthropic publicly said it continued good-faith conversations and disputed some characterizations; reporting cited anonymous senior Pentagon officials and referenced prior classified use of Claude.',
  'sources': [{'url': 'https://www.nbcnews.com/tech/security/anthropic-pentagon-us-military-can-use-ai-missile-defense-hegseth-rcna260534',
    'name': 'NBC News'}],
  'news_date': '2026-02-25',
  'topic': 'Report',
  'created_at': '2026-02-27T00:20:38.763735+00:00'},
 {'event_id': '1647_2026-02-25',
  'output': "Recon Analytics artic

In [32]:
def openai_research_workflow(research_input):
    for i in research_input:
        response = client.responses.create(
            model = "gpt-5-nano",
            tools = [{
                "type": "web_search"
            }],
            include = ["web_search_call.action.sources"],
            input = f"""
            You are a senior research analyst for Krux.

            TASK:
              Create a deep, fact-checked brief for ONE AI report/paper/benchmark release.
            
            GOAL:
              Extract the highest-value findings and implementation guidance so a later 100-word summary can capture most practical signal.
            
            OUTPUT FORMAT:
              - 12 to 16 bullet points.
              - Each bullet must include inline citations:
                [Source: <publisher>, <url>]
              - No text before or after bullets.
            
            MANDATORY STRUCTURE:
              1) REPORT SNAPSHOT
              - What was published, by whom, and when.
              - Scope: geography, sectors, sample size, time window, method type.
            
              2) CORE FINDINGS (MOST IMPORTANT)
              - 4 to 6 bullets with the strongest quantified findings.
              - Include exact numbers, not vague language.
            
              3) WHAT THIS ACTUALLY MEANS
              - 2 to 3 bullets translating findings into plain English implications for real teams.
            
              4) IMPLEMENTATION PLAYBOOK
              - 3 to 4 bullets: what organizations should do in the next quarter.
              - Include sequencing (e.g., data readiness -> pilot -> measurement -> rollout).
            
              5) METRICS TO TRACK
              - 1 or 2 bullets on KPIs teams should monitor to apply the report in practice.
            
              6) LIMITATIONS / CAVEATS
              - 1 or 2 bullets on methodology limits, sample bias, missing data, or uncertainty.
            
              7) DECISION TAKEAWAY
              - 1 bullet: “If a team only does one thing based on this report, it should be X.”
            
              STRICT RULES:
              - No hype, no opinion, no generic AI commentary.
              - Every factual claim must be source-backed.
              - If a key detail is missing, write: "Not disclosed in cited sources."
              - Prefer primary sources (original report/paper) over secondary articles.
              - If secondary coverage conflicts with primary report, prioritize primary and note conflict.
            
              QUALITY CHECK:
              - Must contain quantified findings.
              - Must contain actionable implementation steps.
              - Must clearly separate findings from interpretation.
              - Must include limitations.
            
              Report/event to research:
              Report summary: {i['output']}
              Source: {i['sources']}
            """,
        )

        output = response.output_text
        print(output)

        final_dictionary = {
            'event_id': i['event_id'],
            'news_date': i['news_date'],
            'output': output,
            'model_provider': 'openai',
            'topic': i['topic']
        }
        print(final_dictionary)

        save_research(final_dictionary)

        print("✅ Done")

In [33]:
openai_research_workflow(report_deep_dive)

- REPORT SNAPSHOT — What was published, by whom, and when: Multiple outlets reported that US Defense Secretary Pete Hegseth pressed Anthropic CEO Dario Amodei to loosen Claude usage terms for Pentagon use, with a Friday deadline (February 27, 2026) and potential consequences tied to a $200 million DoD contract; coverage cited Washington Post and Al Jazeera (Reuters/AP), among others. [Source: The Washington Post, https://www.washingtonpost.com/technology/2026/02/24/pentagon-demands-ai-access/] [Source: Al Jazeera, https://www.aljazeera.com/news/2026/2/25/anthropic-vs-the-pentagon-why-ai-firm-is-taking-on-trump-administration]  
- REPORT SNAPSHOT — Scope: geography, sectors, sample size, time window, method type: United States–focused defense/AI procurement context; DoD contracts awarded to four AI firms (Anthropic, Google, OpenAI, xAI) worth up to $200 million each; reporting dates span Feb 24–25, 2026, based on anonymous officials and formal DoD statements. [Source: The Washington Pos

In [8]:
def save_research(research_json):
    supabase.table('research_assistant').insert({
        'event_id': research_json['event_id'],
        'model_provider': research_json['model_provider'],
        'news_date': research_json['news_date'],
        'output': research_json['output'],
        'topic': research_json['topic']
    }).execute()